# 📍 Pose Metric Evaluation (ATE, RTE, RPE)
이 노트북 파일은 `evo` 라이브러리를 활용하여 3DGS 및 LongSplat의 카메라 궤적 예측 결과를 실제 카메라 궤적(GT)과 비교하여 추정 성능(ATE, RTE, RPE)을 평가합니다.
- **ATE (Absolute Trajectory Error)**: 전역 궤적의 절대 위치 오차
- **RTE (Relative Translation Error)**: 상대 병진 위치 오차
- **RPE (Relative Pose Error / RRE)**: 상대 회전 방향 오차 (각도 단위)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install evo pandas tqdm

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 4.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.5/139.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.7 MB/s eta 0:00:00


In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import evo
    from evo.core.trajectory import PoseTrajectory3D
    from evo.tools import file_interface
    from evo.core import sync
    from evo.core import metrics
    from evo.core.metrics import PoseRelation
except ImportError:
    print("Please install evo (e.g. !pip install evo) to compute ATE, RPE.")

In [4]:
def find_trajectory_file(base_dir):
    """
    구조화된 경로 내에서 카메라 궤적 파일(예: trajectory.txt, cameras.txt 등)을 자동으로 탐색합니다.
    """
    # 1. 파일 이름 패턴 검색 (프로젝트 환경에서 출력되는 궤적 파일명에 맞게 확장자를 수정하세요)
    search_patterns = [
        os.path.join(base_dir, "**", "*traj*.txt"),
        os.path.join(base_dir, "**", "poses*.txt"),
        os.path.join(base_dir, "**", "cameras.txt"),
        os.path.join(base_dir, "**", "images.txt")  # COLMAP 형식의 예시
    ]
    
    matches = []
    for pattern in search_patterns:
        matches.extend(glob.glob(pattern, recursive=True))

    if matches:
        # 여러 개가 발견될 경우 가장 하위 경로 반환 (혹은 가장 최신)
        return sorted(matches)[-1]

    return None

In [5]:
def compute_pose_metrics(gt_file, pred_file, align=True):
    """
    GT 궤적과 예측 궤적 간의 ATE, RTE, RPE를 계산합니다.
    (이 기본 함수는 evo 라이브러리와 TUM 궤적 포맷을 기준으로 작성되었습니다.)
    TUM 포맷: [timestamp tx ty tz qx qy qz qw]
    """
    try:
        # 파일 로드 (Kitti 포맷인 경우 read_kitti_poses_file, COLMAP인 경우 evo 내장 변환기 등 적용 필요)
        # COLMAP 텍스트 포맷을 직접 읽으려면 evo_traj colmap 등의 변환 로직 추가 변환 필요할 수 있음
        traj_ref = file_interface.read_tum_trajectory_file(gt_file)
        traj_est = file_interface.read_tum_trajectory_file(pred_file)
    except Exception as e:
        raise ValueError(f"Trajectory 로드 실패. 궤적 파일 포맷에 맞게 수정이 필요할 수 있습니다: {e}")

    # 타임스탬프 기반 궤적 동기화 (만약 타임스탬프가 없다면 인덱스 기반 매칭 필요)
    max_diff = 0.01
    traj_ref, traj_est = sync.associate_trajectories(traj_ref, traj_est, max_diff=max_diff)
    
    if traj_est.num_poses == 0:
        raise ValueError("GT와 PRED 궤적 간에 동기화되는 포즈가 없습니다.")
        
    # Scale 및 Pose Alignment 적용 (Umeyama alignment - 3D 모노큘러 카메라에 필수적)
    if align:
        traj_est.align(traj_ref, correct_scale=True, correct_only_scale=False)
        
    # 1. ATE (Absolute Trajectory Error) - 절대 경로 오차 (위치 차이)
    ape_metric = metrics.APE(PoseRelation.translation_part)
    ape_metric.process_data((traj_ref, traj_est))
    ate_rmse = ape_metric.get_statistic(metrics.StatisticsType.rmse)
    
    # 2. RTE (Relative Translation Error) - 상대 병진 오차 (1 프레임 이동 기준)
    rpe_trans_metric = metrics.RPE(PoseRelation.translation_part, delta=1, delta_unit=metrics.Unit.frames)
    rpe_trans_metric.process_data((traj_ref, traj_est))
    rte_rmse = rpe_trans_metric.get_statistic(metrics.StatisticsType.rmse)
    
    # 3. RPE (Relative Pose/Rotation Error) - 상대 회전 오차 (각도 단위)
    rpe_rot_metric = metrics.RPE(PoseRelation.rotation_angle_deg, delta=1, delta_unit=metrics.Unit.frames)
    rpe_rot_metric.process_data((traj_ref, traj_est))
    rre_rmse = rpe_rot_metric.get_statistic(metrics.StatisticsType.rmse)
    
    return ate_rmse, rte_rmse, rre_rmse

In [ ]:
DIR_BASE = '/content/drive/MyDrive/Research/26_Capstone2'

METHODS = ["LongSplat", "3DGS"]
CONDITIONS = ["original", "snow", "de_snow"]

# 평가할 Scene 리스트
SCENES = [
    "grass",
    "pillar",
    "road"
]

# GT 디렉토리 및 트래젝토리 탐색
GT_DIR_LIST = [os.path.join(DIR_BASE, "GT", f"gt_{s}") for s in SCENES]
GT_TRAJ_LIST = [find_trajectory_file(gt) for gt in GT_DIR_LIST]

# PRED 디렉토리 및 트래젝토리 탐색
SCENE_TRAJ_LIST = []
for s in SCENES:
    scene_traj = []
    for m in METHODS:
        for c in CONDITIONS:
            base_p = os.path.join(DIR_BASE, m, f"{s}_{c}")
            scene_traj.append(find_trajectory_file(base_p))
    SCENE_TRAJ_LIST.append(scene_traj)

# 파일 탐색 결과 출력
print("=== GT Trajectory Files ===")
for i, gt in enumerate(GT_TRAJ_LIST):
    print(f"{SCENES[i]}: {gt}")

print("\n=== Prediction Trajectory Files ===")
for i, scene_paths in enumerate(SCENE_TRAJ_LIST):
    print(f"\nScene: {SCENES[i]}")
    idx = 0
    for m in METHODS:
        for c in CONDITIONS:
            print(f"  [{m} - {c}]: {scene_paths[idx]}")
            idx += 1

In [ ]:
results = []

for i, (gt_file, pred_group) in enumerate(zip(GT_TRAJ_LIST, SCENE_TRAJ_LIST)):
    scene_name = SCENES[i]
    print(f"\n==============================")
    print(f"GT: {gt_file}\n")

    if not gt_file or not os.path.exists(gt_file):
        print(f"GT 궤적 파일을 찾을 수 없음, 스킵: {gt_file}")
        continue

    path_idx = 0
    for m_name in METHODS:
        for c_name in CONDITIONS:
            pred_file = pred_group[path_idx]
            path_idx += 1

            print(f"PRED ({m_name} - {c_name}): {pred_file}")

            if not pred_file or not os.path.exists(pred_file):
                print(f"PRED 궤적 파일을 찾을 수 없음, 스킵")
                continue

            try:
                # 지표 계산 실행
                ate_val, rte_val, rre_val = compute_pose_metrics(gt_file, pred_file, align=True)

                results.append({
                    'Scene': scene_name,
                    'Method': m_name,
                    'Condition': c_name,
                    'ATE_RMSE': ate_val,
                    'RTE_RMSE': rte_val,
                    'RRE_RMSE_deg': rre_val
                })
                
                print(f"-> ATE: {ate_val:.4f}, RTE: {rte_val:.4f}, RRE: {rre_val:.4f}°")

            except Exception as e:
                print(f"-> Error: {e}")
                continue

            print()

if results:
    # CSV 저장 및 출력
    df = pd.DataFrame(results)
    csv_path = os.path.join(DIR_BASE, 'pose_metrics_results.csv')
    df.to_csv(csv_path, index=False)
    print(f"\n[DONE] All pose metrics saved to: {csv_path}")
    display(df)
else:
    print("평가된 결과가 없습니다. 궤적 파일(GT 및 PRED) 포맷 또는 경로를 확인하세요.")